# 2. Structure of the HDF5 output file

PlatoSim produces three output files:

* an **HDF5** file which is the subject of this notebook
* a **log** file which contains all info, warning, error, and debugging messages
* a copy of the **YAML** input file

We will assume in this notebook that you run PlatoSim and inspect the HDF5 output file from a python environment. The HDF5 file is located in the directory specified in `sim.outputDir`.

<img src="StructureOfPlatoSimHDF5.png">

The group `InputParameters` contains a copy of the configuration parameters from the YAML file. Not all parameters however have yet made it into the HDF5 output file, on-going work.

The group `StarPositions` contains - for each exposure - the pixel coordinates, planar focal plane coordinates, and the flux of the star with starID. The coordinates are averaged positions of the star over the duration of the exposure. Only the stars that fall within the subField during the exposure are stored, i.e. some stars may move out of the subField due to the spacecraft jitter.

The group `StarCatalog` contains the sky coordinates, the pixel coordinates, and the focal plane coordinates of all the stars that were detected during any exposure. The starIDs map the ID from the input starCatalog that was supplied with the configuration. Pixel coordinates and planar focal plane coordinates are initial values before any Jitter takes place.

In this tutorial we show how to inspect and extract information from the HDF5 output file produced by PlatoSim. We present how to use **h5py**, the in-build **h5** function, but we will place the effort on explaining how to fully utilise the in-build **SimFile** class.

Note: apart from the libraries show in this tutorial, it is also possible to investigate HDF(5) files with [PyTables](https://www.pytables.org/index.html), however, this packages is known to course problems with other packages used in PlatoSim and hence we avoid it for our dependencies.

### Setup notebook

In [ ]:
# Reload code outside notebook
%load_ext autoreload
%autoreload 2

# Configure figures in notebook
%matplotlib notebook

### Imports

In [ ]:
import os
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt

# PlatoSim
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_notebook
setup_notebook()

### Run a default simulation for the tutorial

In [ ]:
# Define inputs and outputs
outputDir      = os.getcwd()
outputFileName = "output_example1"
outputFile     = f"{outputDir}/{outputFileName}.hdf5"

# Generate simulation object and control the output
sim = Simulation(outputFileName, outputDir=outputDir)
sim["Camera/IncludeExtendedGhosts"]              = True
sim["ControlHDF5Content/WriteHighResolutionPSF"] = True
sim["ControlHDF5Content/WriteSubPixelImages"]    = True

# Run PlatoSim (and allow to overwrite output file)
sim.run(removeOutputFile=True);

---
## 2.1 - Inspect with [h5py](https://docs.h5py.org/en/stable/)
---

We here show a minimal example of using [h5py](https://docs.h5py.org/en/stable/) to inspect your HDF5 output file. E.g. let's fetch the first exposure simulated:

In [ ]:
import h5py
h5file = h5py.File(outputFile, 'r')

In [ ]:
# Get an image
im = h5file['Images/image000000'][:]
im

---
## 2.2 - Inspect with h5
---

PlatoSim has an build-in functionality to use `h5py` to print the structure and fetch the information from the HDF5 output file:

In [ ]:
from platosim.h5 import h5get, h5ls

### Function: h5ls

The function **h5ls** takes an HDF5 file object or a HDF5 group as a mandatory argument and shows the complete structure of the HDF5 file or group. Each level is indicated by the following type acronyms, and for attributes their value is shown.

[G] Group <br/>
[D] Dataset <br/>
[a] Attribute

Print the entire HDF5 file - equivalent to specifying only the root group `h5ls(h5file['/'])`:

In [ ]:
h5ls(h5file)

In [ ]:
# Example of a Dataset
h5ls(h5file['BiasMapsLeft'])

In [ ]:
# Example of attributes
h5ls(h5file['InputParameters/CCD/Gain'])

### Function: h5get

With **h5get** you can get data out of the HDF5 file into numpy arrays or python variables. This function takes two mandatory arguments, the HDF5 file object (or group) and the 'path into the variable or dataset'.

When you specify the full path, only that variable is returned as shown in the following two commands. When you specify a partial path or just the name of the final dataset/attribute, the **h5get** function looks for all possible matches and returns an array with their values. This is illustrated further with 'ReadoutNoise'.

In [ ]:
pos = h5get(h5file, "/InputParameters/CCD/Position", verbose=False)
print ("Type and value of Position: {}, {}".format(type(pos), pos))

In [ ]:
noise = h5get(h5file, "/InputParameters/CCD/ReadoutNoise", verbose=False)
print ("Type and value of ReadoutNoise: {}, {}".format(type(noise), noise))

In [ ]:
h5get(h5file, "ReadoutNoise")

In [ ]:
cols = h5get(h5file, "InputParameters/CCD/NumColumns", verbose=False)
print ("Type and value of NumColumns: {}, {}".format(type(cols), cols))

In [ ]:
dec = h5get(h5file, "ACS/PlatformDec", verbose=False)
ra  = h5get(h5file, "ACS/PlatformRA", verbose=False)
print ("Type and shape of RA : {}, {}".format(type(ra), ra.shape))
print ("Type and shape of Dec: {}, {}".format(type(dec), dec.shape))

In [ ]:
im = h5get(h5file,"Images/image000000", verbose=False)
im

---
## 2.3 - Inspect with SimFile
---

In the previous tutorials we already touched upon how you can use the PlatoSim `SimFile` class to retrieve information from the HDF5 output file. Here we dive in a bit deeper and show must of its functionalites. First let's get the HDF5 file with:

In [ ]:
f = SimFile(outputFileName + ".hdf5")

`reload()`: If you accidentially overwrites/mess up some parameters, you can always reload the HDF5 output again using: 

In [ ]:
f.reload()

For the sake of this tutorial we will only inspect the first exposure, hence:

In [ ]:
imageNr = 0

### Input Parameters

`getInputParameter()`: We here give a minimal example to show how to inspect one of the input parameters that was used for the simulation. The group name and parameter name are exactly the same as in the YAML file. E.g. fetch the `CycleTime`:

In [ ]:
f.getInputParameter("ObservingParameters", "CycleTime")

`getExposureTime()`: Get the exposure time:

In [ ]:
t_exp = f.getExposureTime()
t_exp

`getReadoutTime()`: Get the readout time before the next exposure starts and the readout time during the next exposure. Note that this function automatically access which camera (*Normal* or *Fast*) has been used in the simulation:

In [ ]:
t_out_before, t_out_after = f.getReadoutTime()
t_out_before, t_out_after

``getTime()``: Get time column of all exposures:

In [ ]:
time = f.getTime()[:]
time

### Pointing, Jitter, and Drift

`getPlatformPointingCoordinates()`: Get the platform jitter in either equatorial coordinates (RA, Dec):

In [ ]:
RA, Dec = f.getPlatformPointingCoordinates()

In [ ]:
plt.figure(figsize=(6,5))
plt.plot(RA, Dec, 'o--')
plt.xlabel("RA")
plt.ylabel("Dec")
plt.tight_layout()
plt.show()

 `getYawPitchRoll()`: Or in the spacecraft rotation angles (yaw, pitch, roll):

In [ ]:
yaw, pitch, roll = f.getYawPitchRoll()

In [ ]:
plt.figure(figsize=(7,4))
exp = np.arange(1,11,1)
plt.plot(exp, yaw,   'o--', label="Yaw")
plt.plot(exp, pitch, 'o--', label="Pitch")
plt.plot(exp, roll,  'o--', label="Roll")
plt.xlabel("Exposure Nr")
plt.ylabel("Amplitude [arcsec]")
plt.tight_layout()
plt.legend()
plt.show()

### Background and Stray light

`getPointLikeGhostCoordinates()`: Get information about the point-like ghosts:

In [ ]:
ID, row, col, Xmm, Ymm, flux = f.getPointLikeGhostCoordinates(imageNr)
ID

`getExtendedGhostCoordinates()`: Get information about extended ghosts:

In [ ]:
ID, row, col, Xmm, Ymm, flux, radius = f.getExtendedGhostCoordinates(imageNr)
ID

`getCosmicsInfo()`: You can also retrieve information about the cosmic rays added to each exposure. Since cosmic rays can be added to several image products, note that the `field` parameters can determines from what field the Cosmics should be returned: `SubField`, `BiasMapLeft`, `BiasMapRight` or `SmearingMap`. We here fetch the cosmic ray information for exposure 0 in the star image:

In [ ]:
entryRows, entryColumns, entryAngles, intensities, trailLengths = f.getCosmicsInfo(imageNr, field="SubField")
intensities

`getCosmicsAffectedPixels()`: Get knowledge about the pixels that are affected by cosmic rays:

In [ ]:
row, col, flux = f.getCosmicsAffectedPixels(imageNr)
flux

### Stellar information

`getStarCatalog()`: Get the catalogue of stars that have been detected on a CCD:

In [ ]:
ID, RA, dec, Vmag, xFPmm, yFPmm, rowPix, colPix = f.getStarCatalog()
ID

`getStarCoordinates()`: Get the stellar coordinates in focal plane as well (in pixel and mm in the CCD focal plane). It is also possible to only get the coordinates of stars within a certain magnitude range `[minVmag, maxVmag]` as we show in the following for exposre 0:

In [ ]:
ID, row, col, Xmm, Ymm, flux = f.getStarCoordinates(imageNr, minVmag=10.0, maxVmag=12.0)
ID

### Subfield images

`getImage()`: Get the full simulated subfield and the image dimentions and use them as extentions:

In [ ]:
image      = f.getImage(imageNr)
numRows    = f.getInputParameter("SubField", "NumRows")
numColumns = f.getInputParameter("SubField", "NumColumns")

Let's plot the result:

In [ ]:
# Simple plot using imshow
plt.figure(figsize=(6,6))
plt.imshow(image, cmap=cm.hot, interpolation="nearest", origin='lower', extent=[0, numRows, 0, numColumns])
plt.scatter(np.floor(col)+0.5, np.floor(row)+0.5, marker='x', c='g')
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.tight_layout()
plt.show()

Another options is to use the build-in function `showImage()` to visualize your subfield together with the stellar coordinates:

In [ ]:
fig, ax = f.showImage(imageNr, clipPercentile=1, imgScale="clip",
                      minVmag=10.0, maxVmag=12.0,
                      figsize=(8,8), fontSize=15, useTitle=False,
                      showStarPositions=True, showStarIDs=False,
                      colorMap="magma", colorBar=True, showGrid=True) 

`getImagette()`: It is also possible to fetch a smaller subfield (known as an *imagette*) around a given target star. Here we first find all the star IDs and select a subfield with radius of 3 pixels:

In [ ]:
starIDs = f.getStarCoordinates(imageNr, minVmag=None, maxVmag=None)[0]

In [ ]:
imagette = f.getImagette(starIDs[0], imageNr, radius=3)

In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(imagette, interpolation='nearest', origin='lower', cmap="magma")
plt.colorbar(fraction=0.045)
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.tight_layout()
plt.show()

### Point Spred Function (PSF)

`getPSF()`: The PSF can also be fetched. Above we saved the high resolution PSF to file, but it is also possible to save the diffused PSF. Here we fetch the following:

In [ ]:
psf = f.getPSF("HighResPSFmapCenterSubfield")

In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(psf, interpolation='nearest', origin='lower', cmap="magma")
plt.colorbar(fraction=0.045)
plt.xlabel("x [subpixel]")
plt.ylabel("y [subpixel]")
plt.tight_layout()
plt.show()

`showPSF()`: Again there is an in-build function to show the used PSF:

In [ ]:
f.showPSF("HighResPSFmapCenterSubfield", useTitle=False)

### Pixel maps

`getSmearingMap()`: The smearing map (from a parallel overscan) can be fetched with:

In [ ]:
sm = f.getSmearingMap(imageNr)

In [ ]:
plt.figure(figsize=(8,3))
plt.imshow(sm, interpolation='nearest', origin='lower', cmap="magma")
plt.colorbar(fraction=0.015)
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.tight_layout()
plt.show()

The bias map (from a serial prescan) can be fetched with for the left CCD half with `getBiasMapLeft()` and for the right CCD half with `getBiasMapRight()`. Here we show the former:

In [ ]:
bm = f.getBiasMapLeft(imageNr)

In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(bm, interpolation='nearest', origin='lower', cmap="magma")
plt.colorbar(fraction=0.045)
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.tight_layout()
plt.show()

 `getThroughputMap()`: Get the throughput map:

In [ ]:
tm = f.getThroughputMap(imageNr)

In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(tm, interpolation='nearest', origin='lower', cmap="magma")
plt.colorbar(fraction=0.045)
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.tight_layout()
plt.show()

`getPRNU()`: Get the Pixel Response Non-Uniformity (PRNU; which essentially is the flat-field image):

In [ ]:
prnu = f.getPRNU()

In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(prnu, interpolation='nearest', origin='lower', cmap="magma")
plt.colorbar(fraction=0.045)
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.tight_layout()
plt.show()

In [ ]:
# Do not exist at the moment!
# irnu = f.getIRNU()
# plt.imshow(irnu, interpolation='nearest', cmap=cm.hot)

#             >>> file = SimFile("Simul01.hdf5")                                                                                                                                                              
#             >>> IRNU = file.getIRNU()                                                                                                                                                                       
#             >>> plt.imshow(IRNU, cmap="hot", interpolation="nearest")                                                                                                                                       
#             >>> plt.colorbar()                                                                                                                                                                              
                                                                                                                                                                                                            
#             To rebin the IRNU to the PRNU:                                                                                                                                                                  
                                                                                                                                                                                                            
#             >>> Nrows, Ncols = 100, 100      # size in pixels of the subfield                                                                                                                               
#             >>> NsubPixels = 16              # 16^2 subpixels in 1 pixel                                                                                                                                    
#             >>> assert(IRNU.shape == (Nrows*NsubPixels, Ncols*NsubPixels))                                                                                                                                  
#             >>> PRNU = IRNU.reshape(Nrows, NsubPixels, Ncols, NsubPixels).sum(axis=3).sum(axis=1) 